# Stage 8: NTHUDDD2 Kaggle Extracted-Frame Training

This notebook trains three full-frame CNN baselines for the **NTHUDDD2 Kaggle extracted-frame JPG branch**:

- CNN-1: ResNet18
- CNN-2: MobileNetV2
- CNN-3: EfficientNet-B0

This is a **single-frame binary image-classification baseline**:

- `notdrowsy = 0`
- `drowsy = 1`

Important scope notes:

- This is the **Kaggle extracted-frame JPG version** of NTHUDDD2.
- This is **not** the official raw-video NTHU-DDD benchmark protocol.
- This notebook does not redo local preprocessing.
- This notebook does not extract frames from videos.
- This notebook does not perform MTCNN face cropping.
- This notebook does not permanently resize or overwrite original JPG files.
- Images are dynamically resized to `224 x 224` during training.
- LOSO folds exist as follow-up sensitivity analysis, but this first notebook trains only the fixed subject-level split.

## Mount Google Drive

The notebook uses the standard Colab Google Drive mount. It does not use any Drive connector.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Imports and Configuration

All outputs are written under `outputs/nthuddd2_kaggle/` so this notebook does not write into the existing YawDD+ output folders.

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import shutil
import time
import zipfile
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = None

# Re-extract local runtime data even if counts already match.
FORCE_REEXTRACT = False

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
MAX_EPOCHS = 8
FREEZE_BACKBONE_EPOCHS = 1
EARLY_STOPPING_PATIENCE = 2
LEARNING_RATE = 1e-4
NUM_WORKERS = 2

# Diagnostic workflow controls. Defaults preserve original ResNet18 artifacts
# and prevent full three-model training from running on a notebook Run all.
RUN_ORIGINAL_RESNET18 = False
RUN_RESNET18_THRESHOLD_TUNING = True
RUN_RESNET18_V2 = True
RUN_RESNET18_V2_THRESHOLD_TUNING = True
RUN_MOBILENET_V2 = False
RUN_EFFICIENTNET_B0 = False
RUN_WRITE_INITIAL_SUMMARY = False


EXPECTED_COUNTS = {
    "drowsy": 36030,
    "notdrowsy": 30491,
    "total": 66521,
    "train": 40949,
    "val": 18833,
    "test": 6739,
}
EXPECTED_SUBJECTS = {
    "train": {"001", "005"},
    "val": {"002"},
    "test": {"006"},
}
CLASS_TO_ID = {"notdrowsy": 0, "drowsy": 1}
ID_TO_CLASS = {0: "notdrowsy", 1: "drowsy"}

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Drowsiness_Detection_Colab")
DRIVE_DATA_ROOT = DRIVE_PROJECT_ROOT / "data" / "nthuddd2_kaggle" / "train_data"
DRIVE_MANIFEST_ROOT = DRIVE_PROJECT_ROOT / "manifests" / "nthuddd2_kaggle"
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "outputs" / "nthuddd2_kaggle"

DRIVE_DROWSY_ZIP = DRIVE_DATA_ROOT / "drowsy.zip"
DRIVE_NOTDROWSY_ZIP = DRIVE_DATA_ROOT / "notdrowsy.zip"
DRIVE_MANIFEST_PATH = (
    DRIVE_MANIFEST_ROOT / "nthuddd2_kaggle_all_images_trainable_with_split.csv"
)
DRIVE_SUBJECT_SPLIT_PATH = DRIVE_MANIFEST_ROOT / "nthuddd2_kaggle_subject_split.csv"
DRIVE_LOSO_FOLDS_PATH = DRIVE_MANIFEST_ROOT / "nthuddd2_kaggle_loso_folds.csv"

LOCAL_ROOT = Path("/content/nthuddd2_kaggle_runtime")
LOCAL_ZIP_DIR = LOCAL_ROOT / "zips"
LOCAL_DATA_DIR = LOCAL_ROOT / "data" / "train_data"

RESULTS_DIR = DRIVE_OUTPUT_ROOT / "results"
REPORTS_DIR = DRIVE_OUTPUT_ROOT / "reports"
FIGURES_DIR = DRIVE_OUTPUT_ROOT / "figures"
CHECKPOINTS_DIR = DRIVE_OUTPUT_ROOT / "checkpoints"

for folder in [RESULTS_DIR, REPORTS_DIR, FIGURES_DIR, CHECKPOINTS_DIR, LOCAL_ZIP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"


def seed_everything(seed: int = 42) -> None:
    """Seed Python, NumPy, and PyTorch for reproducible Colab runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


print(f"Device: {DEVICE}")
if DEVICE.type != "cuda":
    print(
        "CUDA is not available. The notebook can run on CPU, but GPU runtime is strongly recommended."
    )
print(f"Drive project root: {DRIVE_PROJECT_ROOT}")
print(f"Drive output root: {DRIVE_OUTPUT_ROOT}")

## Drive and Local Path Setup

The two class zip files are copied from Drive to `/content/` and extracted locally for faster I/O. Repeated execution is supported: if local counts match the expected dataset counts and `FORCE_REEXTRACT = False`, the existing local extraction is reused.

In [ ]:
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Re-extract local runtime data even if counts already match.
# Leave False for normal repeated execution; set True manually if you want a clean rebuild.
FORCE_REEXTRACT = False

EXPECTED_COUNTS = {
    "drowsy": 36030,
    "notdrowsy": 30491,
    "total": 66521,
}

ZIP_BY_LABEL = {
    "drowsy": DRIVE_DROWSY_ZIP,
    "notdrowsy": DRIVE_NOTDROWSY_ZIP,
}

LOCAL_ZIP_DIR = LOCAL_ROOT / "zips"
LOCAL_TMP_EXTRACT_DIR = LOCAL_ROOT / "_extract_tmp"


def count_jpg_files(path: Path, recursive: bool = True) -> int:
    if not path.exists():
        return 0
    iterator = path.rglob("*") if recursive else path.glob("*")
    return sum(
        1 for p in iterator if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg"}
    )


def list_jpg_files(path: Path):
    if not path.exists():
        return []
    return [
        p
        for p in path.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".jpg", ".jpeg"}
        and "__MACOSX" not in p.parts
    ]


def reset_local_runtime():
    if LOCAL_DATA_DIR.exists():
        print(f"Removing old local data directory: {LOCAL_DATA_DIR}")
        shutil.rmtree(LOCAL_DATA_DIR)

    if LOCAL_TMP_EXTRACT_DIR.exists():
        print(f"Removing old temp extraction directory: {LOCAL_TMP_EXTRACT_DIR}")
        shutil.rmtree(LOCAL_TMP_EXTRACT_DIR)

    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_ZIP_DIR.mkdir(parents=True, exist_ok=True)


def load_expected_filenames_from_manifest():
    manifest = pd.read_csv(DRIVE_MANIFEST_PATH)

    required_cols = {"filename", "label", "class_id", "split", "subject_id"}
    missing_cols = required_cols - set(manifest.columns)
    if missing_cols:
        raise ValueError(f"Manifest is missing required columns: {missing_cols}")

    expected = {}
    for label in ["drowsy", "notdrowsy"]:
        names = set(
            manifest.loc[manifest["label"] == label, "filename"].astype(str).tolist()
        )
        expected[label] = names
        print(f"Expected filenames from manifest for {label}: {len(names)}")

    assert len(expected["drowsy"]) == EXPECTED_COUNTS["drowsy"], (
        f"Manifest drowsy filename count mismatch: {len(expected['drowsy'])}"
    )
    assert len(expected["notdrowsy"]) == EXPECTED_COUNTS["notdrowsy"], (
        f"Manifest notdrowsy filename count mismatch: {len(expected['notdrowsy'])}"
    )

    return expected


def copy_zip_to_local(label: str, drive_zip: Path) -> Path:
    assert drive_zip.exists(), f"Missing Drive zip file: {drive_zip}"

    local_zip = LOCAL_ZIP_DIR / f"{label}.zip"
    print(f"Copying {drive_zip} -> {local_zip}")
    shutil.copy2(drive_zip, local_zip)
    return local_zip


def extract_label_zip_manifest_safe(label: str, expected_filenames: set):
    drive_zip = ZIP_BY_LABEL[label]
    local_zip = copy_zip_to_local(label, drive_zip)

    label_tmp_dir = LOCAL_TMP_EXTRACT_DIR / label
    label_out_dir = LOCAL_DATA_DIR / label

    if label_tmp_dir.exists():
        shutil.rmtree(label_tmp_dir)
    if label_out_dir.exists():
        shutil.rmtree(label_out_dir)

    label_tmp_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting {local_zip.name} into temp directory for label={label}...")
    with zipfile.ZipFile(local_zip, "r") as z:
        z.extractall(label_tmp_dir)

    extracted_jpgs = list_jpg_files(label_tmp_dir)
    print(f"Raw extracted JPG/JPEG files for {label}: {len(extracted_jpgs)}")

    # Map by filename. If the zip contains duplicate nested copies, keep only one copy.
    filename_to_path = {}
    duplicate_name_count = 0

    for img_path in extracted_jpgs:
        if img_path.name in filename_to_path:
            duplicate_name_count += 1
            continue
        filename_to_path[img_path.name] = img_path

    print(f"Unique JPG filenames found for {label}: {len(filename_to_path)}")
    print(f"Duplicate filename entries ignored for {label}: {duplicate_name_count}")

    missing = sorted(expected_filenames - set(filename_to_path.keys()))
    extra = sorted(set(filename_to_path.keys()) - expected_filenames)

    if missing:
        print("Example missing filenames:", missing[:10])
        raise FileNotFoundError(
            f"{label}: {len(missing)} expected manifest files were not found in zip extraction."
        )

    if extra:
        print(f"{label}: ignoring {len(extra)} extra files not listed in manifest.")
        print("Example extra filenames:", extra[:10])

    print(f"Copying manifest-matched {label} images into final local class folder...")
    for filename in sorted(expected_filenames):
        src = filename_to_path[filename]
        dst = label_out_dir / filename
        shutil.copy2(src, dst)

    final_direct_count = count_jpg_files(label_out_dir, recursive=False)
    final_recursive_count = count_jpg_files(label_out_dir, recursive=True)

    print(f"Final direct JPG count for {label}: {final_direct_count}")
    print(f"Final recursive JPG count for {label}: {final_recursive_count}")

    assert final_direct_count == EXPECTED_COUNTS[label], (
        f"Expected {EXPECTED_COUNTS[label]} direct {label} images, found {final_direct_count}"
    )
    assert final_recursive_count == EXPECTED_COUNTS[label], (
        f"Expected {EXPECTED_COUNTS[label]} recursive {label} images, found {final_recursive_count}. "
        "This means nested duplicates still exist."
    )


def prepare_local_data_manifest_safe(force_reextract: bool = False):
    current_drowsy_direct = count_jpg_files(LOCAL_DATA_DIR / "drowsy", recursive=False)
    current_notdrowsy_direct = count_jpg_files(
        LOCAL_DATA_DIR / "notdrowsy", recursive=False
    )
    current_drowsy_recursive = count_jpg_files(
        LOCAL_DATA_DIR / "drowsy", recursive=True
    )
    current_notdrowsy_recursive = count_jpg_files(
        LOCAL_DATA_DIR / "notdrowsy", recursive=True
    )
    current_total_direct = current_drowsy_direct + current_notdrowsy_direct
    current_total_recursive = current_drowsy_recursive + current_notdrowsy_recursive

    print(
        "Current local direct image counts: "
        f"drowsy={current_drowsy_direct}, notdrowsy={current_notdrowsy_direct}, "
        f"total={current_total_direct}"
    )
    print(
        "Current local recursive image counts: "
        f"drowsy={current_drowsy_recursive}, notdrowsy={current_notdrowsy_recursive}, "
        f"total={current_total_recursive}"
    )

    already_ok = (
        current_drowsy_direct == EXPECTED_COUNTS["drowsy"]
        and current_notdrowsy_direct == EXPECTED_COUNTS["notdrowsy"]
        and current_total_direct == EXPECTED_COUNTS["total"]
        and current_drowsy_recursive == EXPECTED_COUNTS["drowsy"]
        and current_notdrowsy_recursive == EXPECTED_COUNTS["notdrowsy"]
        and current_total_recursive == EXPECTED_COUNTS["total"]
    )

    if already_ok and not force_reextract:
        print(
            "Local extraction already matches expected direct and recursive counts. Reusing local data."
        )
        return

    print("Rebuilding local extracted dataset from Drive zip files...")
    reset_local_runtime()

    expected_filenames = load_expected_filenames_from_manifest()

    extract_label_zip_manifest_safe("drowsy", expected_filenames["drowsy"])
    extract_label_zip_manifest_safe("notdrowsy", expected_filenames["notdrowsy"])

    final_drowsy = count_jpg_files(LOCAL_DATA_DIR / "drowsy", recursive=True)
    final_notdrowsy = count_jpg_files(LOCAL_DATA_DIR / "notdrowsy", recursive=True)
    final_total = final_drowsy + final_notdrowsy

    print("\nFinal local extraction counts:")
    print("drowsy:", final_drowsy)
    print("notdrowsy:", final_notdrowsy)
    print("total:", final_total)

    assert final_drowsy == EXPECTED_COUNTS["drowsy"], (
        f"Expected {EXPECTED_COUNTS['drowsy']} drowsy images, found {final_drowsy}"
    )
    assert final_notdrowsy == EXPECTED_COUNTS["notdrowsy"], (
        f"Expected {EXPECTED_COUNTS['notdrowsy']} notdrowsy images, found {final_notdrowsy}"
    )
    assert final_total == EXPECTED_COUNTS["total"], (
        f"Expected {EXPECTED_COUNTS['total']} total images, found {final_total}"
    )

    # Clean temporary extraction directory to save local disk.
    if LOCAL_TMP_EXTRACT_DIR.exists():
        print(f"Removing temporary extraction directory: {LOCAL_TMP_EXTRACT_DIR}")
        shutil.rmtree(LOCAL_TMP_EXTRACT_DIR)

    print("Local extraction sanity checks passed.")


if "seed_everything" not in globals():

    def seed_everything(seed: int = 42) -> None:
        """Fallback seed helper if this cell is rerun independently after a runtime reset."""
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True


seed_everything(SEED)
prepare_local_data_manifest_safe(FORCE_REEXTRACT)

## Load Manifests and Rebuild Runtime Image Paths

The original `image_path` column was produced locally outside Colab, so this notebook reconstructs runtime paths using `LOCAL_DATA_DIR / label / filename`.

In [ ]:
def normalize_subject_id(value) -> str:
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    return text.zfill(3)


def load_and_validate_manifest() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    manifest = pd.read_csv(DRIVE_MANIFEST_PATH, dtype={"subject_id": str})
    subject_split = pd.read_csv(DRIVE_SUBJECT_SPLIT_PATH, dtype={"subject_id": str})
    loso_folds = pd.read_csv(DRIVE_LOSO_FOLDS_PATH, dtype={"subject_id": str})

    required_cols = {
        "image_path",
        "relative_path",
        "filename",
        "label",
        "class_id",
        "subject_id",
        "condition",
        "state_token",
        "frame_index",
        "source_dataset",
        "image_ok",
        "width",
        "height",
        "split",
    }
    missing = sorted(required_cols - set(manifest.columns))
    if missing:
        raise ValueError(f"Manifest missing required columns: {missing}")

    manifest["subject_id"] = manifest["subject_id"].map(normalize_subject_id)
    subject_split["subject_id"] = subject_split["subject_id"].map(normalize_subject_id)
    loso_folds["subject_id"] = loso_folds["subject_id"].map(normalize_subject_id)
    manifest["class_id"] = manifest["class_id"].astype(int)
    manifest["runtime_image_path"] = manifest.apply(
        lambda row: str(LOCAL_DATA_DIR / str(row["label"]) / str(row["filename"])),
        axis=1,
    )

    assert len(manifest) == EXPECTED_COUNTS["total"], (
        f"Manifest rows mismatch: {len(manifest)}"
    )
    assert set(manifest["label"].unique()) == {"drowsy", "notdrowsy"}, (
        "Unexpected labels in manifest"
    )
    observed_class_map = manifest.groupby("label")["class_id"].nunique().to_dict()
    assert observed_class_map == {"drowsy": 1, "notdrowsy": 1}, (
        f"Unexpected label/class mapping counts: {observed_class_map}"
    )
    for label, class_id in CLASS_TO_ID.items():
        got = set(manifest.loc[manifest["label"] == label, "class_id"].unique())
        assert got == {class_id}, f"Expected {label}={class_id}, got {got}"

    split_counts = manifest["split"].value_counts().to_dict()
    for split, expected in [("train", 40949), ("val", 18833), ("test", 6739)]:
        assert split_counts.get(split, 0) == expected, (
            f"Expected {split}={expected}, got {split_counts.get(split, 0)}"
        )

    for split, expected_subjects in EXPECTED_SUBJECTS.items():
        got = set(manifest.loc[manifest["split"] == split, "subject_id"].unique())
        assert got == expected_subjects, (
            f"Expected {split} subjects {expected_subjects}, got {got}"
        )

    subject_split_counts = manifest.groupby("subject_id")["split"].nunique()
    assert (subject_split_counts == 1).all(), (
        "Subject leakage detected: at least one subject appears in multiple splits"
    )

    for split in ["train", "val", "test"]:
        got_classes = set(manifest.loc[manifest["split"] == split, "class_id"].unique())
        assert got_classes == {0, 1}, (
            f"Split {split} does not contain both classes: {got_classes}"
        )

    missing_paths = [p for p in manifest["runtime_image_path"] if not Path(p).is_file()]
    if missing_paths:
        preview = missing_paths[:5]
        raise FileNotFoundError(
            f"Missing {len(missing_paths)} runtime image files. Examples: {preview}"
        )

    print("Manifest sanity checks passed.")
    print("Split counts:")
    print(manifest.groupby("split").size().reindex(["train", "val", "test"]))
    print("Subject assignments:")
    print(
        manifest.groupby("split")["subject_id"]
        .unique()
        .reindex(["train", "val", "test"])
    )
    print("Class counts by split:")
    print(
        pd.crosstab(manifest["split"], manifest["label"]).reindex(
            ["train", "val", "test"]
        )
    )
    return manifest, subject_split, loso_folds


manifest_df, subject_split_df, loso_folds_df = load_and_validate_manifest()

## Dataset Sanity Checks

The fixed split is subject-level: train subjects are `001` and `005`, validation subject is `002`, and test subject is `006`. This prevents identity leakage, but validation and test each contain only one subject.

In [ ]:
print("Dataset summary")
print(f"Total trainable images: {len(manifest_df):,}")
print(manifest_df["label"].value_counts().sort_index())
print("Parsed subjects:", sorted(manifest_df["subject_id"].unique()))
print("Subject split table:")
display(subject_split_df.sort_values(["split", "subject_id"]))
print(
    "LOSO folds are available for follow-up sensitivity analysis, but are not run in this notebook."
)
display(loso_folds_df.head(16))

## PyTorch Dataset and Dataloaders

Training uses random augmentation. Validation, test, and train-evaluation loaders use deterministic transforms.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomRotation(degrees=8),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.RandomAffine(degrees=0, scale=(0.95, 1.05)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


class NTHUDDD2KaggleDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        path = Path(row["runtime_image_path"])
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        label = int(row["class_id"])
        return image, label


def compute_class_weights(train_df: pd.DataFrame) -> torch.Tensor:
    counts = train_df["class_id"].value_counts().sort_index()
    total = counts.sum()
    weights = [total / (len(counts) * counts.get(class_id, 1)) for class_id in [0, 1]]
    return torch.tensor(weights, dtype=torch.float32)


def build_dataloaders(
    manifest: pd.DataFrame,
) -> Tuple[Dict[str, DataLoader], torch.Tensor]:
    train_df = manifest[manifest["split"] == "train"].copy()
    val_df = manifest[manifest["split"] == "val"].copy()
    test_df = manifest[manifest["split"] == "test"].copy()

    loaders = {
        "train": DataLoader(
            NTHUDDD2KaggleDataset(train_df, train_transform),
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda"),
        ),
        "train_eval": DataLoader(
            NTHUDDD2KaggleDataset(train_df, eval_transform),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda"),
        ),
        "val": DataLoader(
            NTHUDDD2KaggleDataset(val_df, eval_transform),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda"),
        ),
        "test": DataLoader(
            NTHUDDD2KaggleDataset(test_df, eval_transform),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda"),
        ),
    }
    class_weights = compute_class_weights(train_df)
    print(
        f"Class weights from train split [notdrowsy, drowsy]: {class_weights.tolist()}"
    )
    return loaders, class_weights


loaders, class_weights = build_dataloaders(manifest_df)

## Model Definitions

Each model uses torchvision ImageNet pretrained weights. The feature extractor is frozen for the first epoch, then unfrozen for fine-tuning.

In [ ]:
def build_model(model_key: str) -> nn.Module:
    if model_key == "resnet18":
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, 2)
    elif model_key == "mobilenet_v2":
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
        model = models.mobilenet_v2(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    elif model_key == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    else:
        raise ValueError(f"Unknown model_key: {model_key}")
    return model


def set_backbone_trainable(model: nn.Module, model_key: str, trainable: bool) -> None:
    for param in model.parameters():
        param.requires_grad = trainable

    if model_key == "resnet18":
        for param in model.fc.parameters():
            param.requires_grad = True
    elif model_key in {"mobilenet_v2", "efficientnet_b0"}:
        for param in model.classifier.parameters():
            param.requires_grad = True
    else:
        raise ValueError(f"Unknown model_key: {model_key}")


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Training Utilities

Early stopping and checkpoint selection use validation F1-score with `drowsy` as the positive class.

In [ ]:
def save_json(data, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        json.dump(data, fh, indent=2, default=to_builtin)


def to_builtin(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


def binary_metrics(
    y_true: List[int], y_pred: List[int], loss: float | None = None
) -> Dict[str, float]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=1,
        zero_division=0,
    )
    metrics = {
        "loss": float(loss) if loss is not None else None,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    return metrics


def train_one_epoch(
    model, loader, criterion, optimizer, scaler
) -> Tuple[float, List[int], List[int]]:
    model.train()
    total_loss = 0.0
    all_true, all_pred = [], []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        all_true.extend(labels.detach().cpu().numpy().tolist())
        all_pred.extend(preds.detach().cpu().numpy().tolist())

    return total_loss / len(loader.dataset), all_true, all_pred


@torch.no_grad()
def evaluate_model(
    model, loader, criterion=None
) -> Tuple[float | None, List[int], List[int]]:
    model.eval()
    total_loss = 0.0
    all_true, all_pred = [], []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels) if criterion is not None else None
        if loss is not None:
            total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        all_true.extend(labels.detach().cpu().numpy().tolist())
        all_pred.extend(preds.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset) if criterion is not None else None
    return avg_loss, all_true, all_pred


def plot_training_curve(history: List[Dict], model_key: str, out_path: Path) -> None:
    epochs = [row["epoch"] for row in history]
    train_loss = [row["train_loss"] for row in history]
    val_loss = [row["val_loss"] for row in history]
    train_f1 = [row["train_f1"] for row in history]
    val_f1 = [row["val_f1"] for row in history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, train_loss, label="train loss")
    axes[0].plot(epochs, val_loss, label="val loss")
    axes[0].set_title(f"{model_key} loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()

    axes[1].plot(epochs, train_f1, label="train F1")
    axes[1].plot(epochs, val_f1, label="val F1")
    axes[1].set_title(f"{model_key} F1")
    axes[1].set_xlabel("epoch")
    axes[1].legend()

    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_confusion(cm: np.ndarray, title: str, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks([0, 1], labels=["notdrowsy", "drowsy"])
    ax.set_yticks([0, 1], labels=["notdrowsy", "drowsy"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def train_model(model_key: str, display_name: str) -> Tuple[List[Dict], Dict]:
    print(f"\n===== Training {display_name} =====")
    seed_everything(SEED)
    model = build_model(model_key).to(DEVICE)
    set_backbone_trainable(model, model_key, trainable=False)
    print(
        f"Trainable parameters during frozen epoch: {count_trainable_parameters(model):,}"
    )

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=1
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_val_f1 = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    checkpoint_path = CHECKPOINTS_DIR / f"{model_key}_best.pt"
    history: List[Dict] = []

    for epoch in range(1, MAX_EPOCHS + 1):
        if epoch == FREEZE_BACKBONE_EPOCHS + 1:
            set_backbone_trainable(model, model_key, trainable=True)
            print(
                f"Unfroze backbone at epoch {epoch}. Trainable parameters: {count_trainable_parameters(model):,}"
            )

        start = time.time()
        train_loss, train_true, train_pred = train_one_epoch(
            model, loaders["train"], criterion, optimizer, scaler
        )
        val_loss, val_true, val_pred = evaluate_model(model, loaders["val"], criterion)
        train_metrics = binary_metrics(train_true, train_pred, train_loss)
        val_metrics = binary_metrics(val_true, val_pred, val_loss)
        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_metrics["f1"])

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_balanced_accuracy": train_metrics["balanced_accuracy"],
            "train_precision": train_metrics["precision"],
            "train_recall": train_metrics["recall"],
            "train_f1": train_metrics["f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "learning_rate": current_lr,
            "seconds": time.time() - start,
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d}: train_loss={train_metrics['loss']:.4f} train_f1={train_metrics['f1']:.4f} "
            f"val_loss={val_metrics['loss']:.4f} val_f1={val_metrics['f1']:.4f} lr={current_lr:.2e}"
        )

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "model_key": model_key,
                    "display_name": display_name,
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_f1": best_val_f1,
                    "config": {
                        "image_size": IMAGE_SIZE,
                        "batch_size": BATCH_SIZE,
                        "max_epochs": MAX_EPOCHS,
                        "freeze_backbone_epochs": FREEZE_BACKBONE_EPOCHS,
                        "learning_rate": LEARNING_RATE,
                        "seed": SEED,
                    },
                },
                checkpoint_path,
            )
            print(f"Saved best checkpoint: {checkpoint_path}")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(
                    f"Early stopping at epoch {epoch}; best epoch was {best_epoch} with val_f1={best_val_f1:.4f}"
                )
                break

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])

    split_metrics = {}
    for split_name, loader_key in [
        ("train", "train_eval"),
        ("val", "val"),
        ("test", "test"),
    ]:
        loss, y_true, y_pred = evaluate_model(model, loaders[loader_key], criterion)
        metrics = binary_metrics(y_true, y_pred, loss)
        split_metrics[split_name] = metrics
        if split_name == "test":
            cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
            metrics["confusion_matrix"] = cm.tolist()
            metrics["classification_report"] = classification_report(
                y_true,
                y_pred,
                labels=[0, 1],
                target_names=["notdrowsy", "drowsy"],
                zero_division=0,
                output_dict=True,
            )
            plot_confusion(
                cm,
                f"{display_name} test confusion matrix",
                FIGURES_DIR / f"{model_key}_test_confusion_matrix.png",
            )

    plot_training_curve(
        history, model_key, FIGURES_DIR / f"{model_key}_training_curve.png"
    )

    metrics_payload = {
        "model_key": model_key,
        "display_name": display_name,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        "checkpoint_path": str(checkpoint_path),
        "metrics": split_metrics,
    }
    save_json(history, RESULTS_DIR / f"{model_key}_history.json")
    save_json(metrics_payload, RESULTS_DIR / f"{model_key}_metrics.json")
    print(f"Finished {display_name}. Test F1={split_metrics['test']['f1']:.4f}")
    return history, metrics_payload


all_histories = {}
all_metrics = {}

## Train ResNet18

In [ ]:
ORIGINAL_RESNET18_METRICS_PATH = RESULTS_DIR / "resnet18_metrics.json"
ORIGINAL_RESNET18_HISTORY_PATH = RESULTS_DIR / "resnet18_history.json"
ORIGINAL_RESNET18_CHECKPOINT_PATH = CHECKPOINTS_DIR / "resnet18_best.pt"


def load_json_if_exists(path: Path):
    if not path.exists():
        print(f"Missing existing artifact: {path}")
        return None
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)


if RUN_ORIGINAL_RESNET18:
    raise RuntimeError(
        "RUN_ORIGINAL_RESNET18 is disabled for this diagnostic notebook to avoid "
        "overwriting the original ResNet18 artifacts. Use a separate output prefix "
        "if you intentionally want to rerun the original experiment."
    )
else:
    print(
        "Loading existing original ResNet18 artifacts; not retraining or overwriting them."
    )
    metrics = load_json_if_exists(ORIGINAL_RESNET18_METRICS_PATH)
    history = load_json_if_exists(ORIGINAL_RESNET18_HISTORY_PATH)
    if metrics is not None:
        all_metrics["resnet18"] = metrics
        print(
            f"Loaded original ResNet18 metrics: best_epoch={metrics.get('best_epoch')}, "
            f"best_val_f1={metrics.get('best_val_f1')}"
        )
    if history is not None:
        all_histories["resnet18"] = history
        print(f"Loaded original ResNet18 history rows: {len(history)}")


## ResNet18 Diagnostic: Original Run

The original ResNet18 run shows clear overfitting after epoch 1. Training F1 rises sharply from `0.7180` at epoch 1 to `0.9358` at epoch 2 and `0.9630` at epoch 3, while validation F1 drops from `0.6781` to `0.6323` and then `0.5916`.

The selected checkpoint has `best_epoch = 1`, which means the chosen model came from the frozen-backbone / classifier-head stage. Fine-tuning with the original settings likely over-adapts to the two training subjects (`001`, `005`) rather than improving transfer to the held-out validation subject (`002`).

This fixed split is subject-limited: validation and test each contain only one subject. The result should be interpreted cautiously as a Kaggle extracted-frame binary image-classification baseline, not as the official raw-video NTHU-DDD benchmark protocol.

Before comparing all three CNNs, ResNet18 should be re-tested with a gentler fine-tuning strategy and validation-only threshold tuning.

In [ ]:
# Threshold tuning uses validation probabilities only to select the threshold.
# The selected threshold is then applied once to the held-out test split.

resnet18_threshold_tuned_metrics = None
resnet18_v2_threshold_tuned_metrics = None


def load_checkpoint_model(
    model_key: str, checkpoint_path: Path
) -> tuple[nn.Module, dict]:
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model = build_model(model_key).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint


@torch.no_grad()
def collect_positive_class_probabilities(
    model: nn.Module, loader: DataLoader
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_true: list[int] = []
    all_prob: list[float] = []
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, 1]
        all_true.extend(labels.numpy().astype(int).tolist())
        all_prob.extend(probs.detach().cpu().numpy().astype(float).tolist())
    return np.asarray(all_true, dtype=int), np.asarray(all_prob, dtype=float)


def metrics_at_threshold(
    y_true: np.ndarray, prob_drowsy: np.ndarray, threshold: float
) -> dict:
    y_pred = (prob_drowsy >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=1,
        zero_division=0,
    )
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "confusion_matrix": cm.tolist(),
        "predicted_notdrowsy": int((y_pred == 0).sum()),
        "predicted_drowsy": int((y_pred == 1).sum()),
    }


def probability_summary(prob_drowsy: np.ndarray) -> dict:
    return {
        "min": float(np.min(prob_drowsy)),
        "p25": float(np.percentile(prob_drowsy, 25)),
        "median": float(np.median(prob_drowsy)),
        "p75": float(np.percentile(prob_drowsy, 75)),
        "max": float(np.max(prob_drowsy)),
        "mean": float(np.mean(prob_drowsy)),
    }


def tune_threshold_for_checkpoint(
    *,
    model_key: str,
    display_name: str,
    checkpoint_path: Path,
    output_path: Path,
    expected_best_epoch: int | None = None,
) -> dict:
    model, checkpoint = load_checkpoint_model(model_key, checkpoint_path)
    checkpoint_epoch = int(checkpoint.get("epoch", -1))
    checkpoint_val_f1 = checkpoint.get("val_f1")
    if expected_best_epoch is not None and checkpoint_epoch != int(expected_best_epoch):
        raise AssertionError(
            f"Checkpoint epoch mismatch for {display_name}: expected best epoch "
            f"{expected_best_epoch}, checkpoint has {checkpoint_epoch}"
        )

    val_true, val_prob = collect_positive_class_probabilities(model, loaders["val"])
    test_true, test_prob = collect_positive_class_probabilities(model, loaders["test"])

    threshold_records = []
    for threshold in np.round(np.arange(0.05, 0.951, 0.01), 2):
        threshold_records.append(
            metrics_at_threshold(val_true, val_prob, float(threshold))
        )

    best_val = max(
        threshold_records,
        key=lambda row: (
            row["f1"],
            row["balanced_accuracy"],
            row["accuracy"],
            -abs(row["threshold"] - 0.5),
        ),
    )
    selected_threshold = float(best_val["threshold"])
    test_metrics = metrics_at_threshold(test_true, test_prob, selected_threshold)

    payload = {
        "display_name": display_name,
        "model_key": model_key,
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_epoch": checkpoint_epoch,
        "checkpoint_val_f1": checkpoint_val_f1,
        "selected_threshold": selected_threshold,
        "validation_f1_at_selected_threshold": float(best_val["f1"]),
        "validation_metrics_at_selected_threshold": best_val,
        "test_metrics_at_selected_threshold": test_metrics,
        "validation_probability_summary": probability_summary(val_prob),
        "test_probability_summary": probability_summary(test_prob),
        "threshold_search": threshold_records,
        "selection_rule": "Maximize validation F1 over thresholds 0.05..0.95 step 0.01; use validation only.",
    }
    save_json(payload, output_path)
    print(f"Saved threshold-tuned metrics: {output_path}")
    print(
        f"{display_name}: selected_threshold={selected_threshold:.2f}, "
        f"val_f1={best_val['f1']:.4f}, test_f1={test_metrics['f1']:.4f}, "
        f"test_bal_acc={test_metrics['balanced_accuracy']:.4f}"
    )
    print("Test confusion matrix [[TN, FP], [FN, TP]]:")
    print(np.asarray(test_metrics["confusion_matrix"]))
    return payload


if RUN_RESNET18_THRESHOLD_TUNING:
    if all_metrics.get("resnet18") is None:
        print(
            "Skipping original ResNet18 threshold tuning because original metrics were not loaded."
        )
    elif not ORIGINAL_RESNET18_CHECKPOINT_PATH.exists():
        print(
            f"Skipping original ResNet18 threshold tuning because checkpoint is missing: {ORIGINAL_RESNET18_CHECKPOINT_PATH}"
        )
    else:
        resnet18_threshold_tuned_metrics = tune_threshold_for_checkpoint(
            model_key="resnet18",
            display_name="ResNet18 original + threshold tuning",
            checkpoint_path=ORIGINAL_RESNET18_CHECKPOINT_PATH,
            output_path=RESULTS_DIR / "resnet18_threshold_tuned_metrics.json",
            expected_best_epoch=all_metrics["resnet18"].get("best_epoch"),
        )
else:
    print(
        "Skipping original ResNet18 threshold tuning. Set RUN_RESNET18_THRESHOLD_TUNING = True to run it."
    )

## ResNet18 v2: Gentler Fine-Tuning

ResNet18 v2 keeps the same fixed subject-level split and binary task, but changes fine-tuning to reduce over-adaptation to train subjects `001` and `005`:

- max epochs: `15`
- early stopping patience: `4`
- epoch 1: frozen backbone, train classifier head only
- after epoch 1: unfreeze backbone
- classifier/head learning rate: `1e-4`
- backbone learning rate after unfreezing: `1e-5`
- checkpoint selection: best validation F1

All v2 artifacts use a `resnet18_v2_*` prefix and do not overwrite original ResNet18 outputs.

In [ ]:
RESNET18_V2_MAX_EPOCHS = 15
RESNET18_V2_PATIENCE = 4
RESNET18_V2_FREEZE_EPOCHS = 1
RESNET18_V2_HEAD_LR = 1e-4
RESNET18_V2_BACKBONE_LR = 1e-5


def resnet18_parameter_groups(
    model: nn.Module, backbone_lr: float, head_lr: float
) -> list[dict]:
    head_params = list(model.fc.parameters())
    head_param_ids = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_param_ids]
    return [
        {"params": backbone_params, "lr": backbone_lr, "name": "backbone"},
        {"params": head_params, "lr": head_lr, "name": "head"},
    ]


def train_resnet18_v2() -> tuple[list[dict], dict]:
    print("\n===== Training ResNet18 v2 =====")
    seed_everything(SEED)
    model = build_model("resnet18").to(DEVICE)
    set_backbone_trainable(model, "resnet18", trainable=False)
    print(
        f"Epoch 1 frozen-backbone trainable parameters: {count_trainable_parameters(model):,}"
    )

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=RESNET18_V2_HEAD_LR
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=1
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_val_f1 = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    checkpoint_path = CHECKPOINTS_DIR / "resnet18_v2_best.pt"
    history: list[dict] = []

    for epoch in range(1, RESNET18_V2_MAX_EPOCHS + 1):
        if epoch == RESNET18_V2_FREEZE_EPOCHS + 1:
            set_backbone_trainable(model, "resnet18", trainable=True)
            optimizer = torch.optim.Adam(
                resnet18_parameter_groups(
                    model, RESNET18_V2_BACKBONE_LR, RESNET18_V2_HEAD_LR
                )
            )
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="max", factor=0.5, patience=1
            )
            print(
                f"Unfroze backbone at epoch {epoch}. "
                f"backbone_lr={RESNET18_V2_BACKBONE_LR:.1e}, head_lr={RESNET18_V2_HEAD_LR:.1e}, "
                f"trainable parameters={count_trainable_parameters(model):,}"
            )

        start = time.time()
        train_loss, train_true, train_pred = train_one_epoch(
            model, loaders["train"], criterion, optimizer, scaler
        )
        val_loss, val_true, val_pred = evaluate_model(model, loaders["val"], criterion)
        train_metrics = binary_metrics(train_true, train_pred, train_loss)
        val_metrics = binary_metrics(val_true, val_pred, val_loss)

        lr_by_group = {
            group.get("name", f"group_{idx}"): group["lr"]
            for idx, group in enumerate(optimizer.param_groups)
        }
        scheduler.step(val_metrics["f1"])

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_balanced_accuracy": train_metrics["balanced_accuracy"],
            "train_precision": train_metrics["precision"],
            "train_recall": train_metrics["recall"],
            "train_f1": train_metrics["f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "learning_rate": max(lr_by_group.values()),
            "learning_rates": lr_by_group,
            "seconds": time.time() - start,
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d}: train_loss={train_metrics['loss']:.4f} train_f1={train_metrics['f1']:.4f} "
            f"val_loss={val_metrics['loss']:.4f} val_f1={val_metrics['f1']:.4f} lr={lr_by_group}"
        )

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "model_key": "resnet18",
                    "display_name": "ResNet18 v2",
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_f1": best_val_f1,
                    "config": {
                        "image_size": IMAGE_SIZE,
                        "batch_size": BATCH_SIZE,
                        "max_epochs": RESNET18_V2_MAX_EPOCHS,
                        "freeze_backbone_epochs": RESNET18_V2_FREEZE_EPOCHS,
                        "head_learning_rate": RESNET18_V2_HEAD_LR,
                        "backbone_learning_rate": RESNET18_V2_BACKBONE_LR,
                        "early_stopping_patience": RESNET18_V2_PATIENCE,
                        "seed": SEED,
                    },
                },
                checkpoint_path,
            )
            print(f"Saved best v2 checkpoint: {checkpoint_path}")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= RESNET18_V2_PATIENCE:
                print(
                    f"Early stopping at epoch {epoch}; best epoch was {best_epoch} with val_f1={best_val_f1:.4f}"
                )
                break

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    if int(checkpoint.get("epoch", -1)) != int(best_epoch):
        raise AssertionError(
            "ResNet18 v2 checkpoint is not the best validation-F1 epoch."
        )
    model.load_state_dict(checkpoint["model_state_dict"])

    split_metrics = {}
    for split_name, loader_key in [
        ("train", "train_eval"),
        ("val", "val"),
        ("test", "test"),
    ]:
        loss, y_true, y_pred = evaluate_model(model, loaders[loader_key], criterion)
        metrics = binary_metrics(y_true, y_pred, loss)
        split_metrics[split_name] = metrics
        if split_name == "test":
            cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
            metrics["confusion_matrix"] = cm.tolist()
            metrics["classification_report"] = classification_report(
                y_true,
                y_pred,
                labels=[0, 1],
                target_names=["notdrowsy", "drowsy"],
                zero_division=0,
                output_dict=True,
            )
            plot_confusion(
                cm,
                "ResNet18 v2 test confusion matrix",
                FIGURES_DIR / "resnet18_v2_test_confusion_matrix.png",
            )

    plot_training_curve(
        history, "resnet18_v2", FIGURES_DIR / "resnet18_v2_training_curve.png"
    )

    metrics_payload = {
        "model_key": "resnet18_v2",
        "display_name": "ResNet18 v2",
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        "checkpoint_path": str(checkpoint_path),
        "metrics": split_metrics,
    }
    save_json(history, RESULTS_DIR / "resnet18_v2_history.json")
    save_json(metrics_payload, RESULTS_DIR / "resnet18_v2_metrics.json")
    print(f"Finished ResNet18 v2. Test F1={split_metrics['test']['f1']:.4f}")
    return history, metrics_payload


if RUN_RESNET18_V2:
    history, metrics = train_resnet18_v2()
    all_histories["resnet18_v2"] = history
    all_metrics["resnet18_v2"] = metrics
else:
    print(
        "Skipping ResNet18 v2. Set RUN_RESNET18_V2 = True to run the gentler fine-tuning experiment."
    )

In [ ]:
if RUN_RESNET18_V2_THRESHOLD_TUNING:
    if all_metrics.get("resnet18_v2") is None:
        print(
            "Skipping ResNet18 v2 threshold tuning because v2 metrics are not available."
        )
    else:
        resnet18_v2_threshold_tuned_metrics = tune_threshold_for_checkpoint(
            model_key="resnet18",
            display_name="ResNet18 v2 + threshold tuning",
            checkpoint_path=CHECKPOINTS_DIR / "resnet18_v2_best.pt",
            output_path=RESULTS_DIR / "resnet18_v2_threshold_tuned_metrics.json",
            expected_best_epoch=all_metrics["resnet18_v2"].get("best_epoch"),
        )
else:
    print(
        "Skipping ResNet18 v2 threshold tuning. Set RUN_RESNET18_V2_THRESHOLD_TUNING = True to run it."
    )

## ResNet18 Diagnostic Comparison

This table compares the original checkpoint, validation-threshold-tuned original checkpoint, ResNet18 v2, and validation-threshold-tuned ResNet18 v2. Thresholds are selected on validation only and then applied once to test.

In [ ]:
def row_from_standard_metrics(label: str, payload: dict | None) -> dict | None:
    if payload is None:
        return None
    test = payload["metrics"]["test"]
    return {
        "model": label,
        "best epoch": payload.get("best_epoch"),
        "selected threshold": "0.50",
        "validation F1": payload.get("best_val_f1"),
        "test accuracy": test.get("accuracy"),
        "test balanced accuracy": test.get("balanced_accuracy"),
        "test precision": test.get("precision"),
        "test recall": test.get("recall"),
        "test F1": test.get("f1"),
        "confusion matrix": test.get("confusion_matrix"),
    }


def row_from_threshold_metrics(label: str, payload: dict | None) -> dict | None:
    if payload is None:
        return None
    test = payload["test_metrics_at_selected_threshold"]
    return {
        "model": label,
        "best epoch": payload.get("checkpoint_epoch"),
        "selected threshold": f"{payload.get('selected_threshold'):.2f}",
        "validation F1": payload.get("validation_f1_at_selected_threshold"),
        "test accuracy": test.get("accuracy"),
        "test balanced accuracy": test.get("balanced_accuracy"),
        "test precision": test.get("precision"),
        "test recall": test.get("recall"),
        "test F1": test.get("f1"),
        "confusion matrix": test.get("confusion_matrix"),
    }


comparison_rows = [
    row_from_standard_metrics("ResNet18 original", all_metrics.get("resnet18")),
    row_from_threshold_metrics(
        "ResNet18 original + threshold tuning", resnet18_threshold_tuned_metrics
    ),
    row_from_standard_metrics("ResNet18 v2", all_metrics.get("resnet18_v2")),
    row_from_threshold_metrics(
        "ResNet18 v2 + threshold tuning", resnet18_v2_threshold_tuned_metrics
    ),
]
comparison_rows = [row for row in comparison_rows if row is not None]
resnet18_diagnostic_comparison_df = pd.DataFrame(comparison_rows)
comparison_csv = RESULTS_DIR / "resnet18_diagnostic_comparison.csv"
resnet18_diagnostic_comparison_df.to_csv(comparison_csv, index=False)
print(f"Saved ResNet18 diagnostic comparison table: {comparison_csv}")
display(resnet18_diagnostic_comparison_df)

## Decision Note

Do not proceed to the full three-model comparison until ResNet18 v2 is reviewed.

- If ResNet18 v2 improves validation/test F1 and reduces overfitting, continue later with MobileNetV2 and EfficientNet-B0 using the v2 training strategy.
- If ResNet18 v2 does not improve much, report the NTHUDDD2 fixed split as subject-limited and consider LOSO evaluation or face-crop/ROI experiments later.
- MobileNetV2 and EfficientNet-B0 are gated by `RUN_MOBILENET_V2 = False` and `RUN_EFFICIENTNET_B0 = False` by default.

## Train MobileNetV2

In [ ]:
if RUN_MOBILENET_V2:
    history, metrics = train_model("mobilenet_v2", "MobileNetV2")
    all_histories["mobilenet_v2"] = history
    all_metrics["mobilenet_v2"] = metrics
else:
    print(
        "Skipping MobileNetV2. Set RUN_MOBILENET_V2 = True only after reviewing ResNet18 v2 diagnostics."
    )


## Train EfficientNet-B0

In [ ]:
if RUN_EFFICIENTNET_B0:
    history, metrics = train_model("efficientnet_b0", "EfficientNet-B0")
    all_histories["efficientnet_b0"] = history
    all_metrics["efficientnet_b0"] = metrics
else:
    print(
        "Skipping EfficientNet-B0. Set RUN_EFFICIENTNET_B0 = True only after reviewing ResNet18 v2 diagnostics."
    )


## Save Metrics, Figures, and Summary Report

This section writes the required CSV, JSON, figures, checkpoints, and markdown summary into `outputs/nthuddd2_kaggle/`.

In [ ]:
def build_results_table(all_metrics: Dict[str, Dict]) -> pd.DataFrame:
    rows = []
    for model_key, payload in all_metrics.items():
        train = payload["metrics"]["train"]
        val = payload["metrics"]["val"]
        test = payload["metrics"]["test"]
        rows.append(
            {
                "CNN Architecture": payload["display_name"],
                "Train Accuracy": train["accuracy"],
                "Validation Accuracy": val["accuracy"],
                "Test Accuracy": test["accuracy"],
                "Balanced Accuracy": test["balanced_accuracy"],
                "Precision": test["precision"],
                "Recall": test["recall"],
                "F1-score": test["f1"],
            }
        )
    return (
        pd.DataFrame(rows)
        .sort_values(["F1-score", "Balanced Accuracy"], ascending=False)
        .reset_index(drop=True)
    )


def format_metric(value: float) -> str:
    return f"{value:.4f}"


def markdown_results_table(results_df: pd.DataFrame) -> str:
    lines = [
        "| CNN Architecture | Train Accuracy | Validation Accuracy | Test Accuracy | Balanced Accuracy | Precision | Recall | F1-score |",
        "|---|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for _, row in results_df.iterrows():
        lines.append(
            f"| {row['CNN Architecture']} | {format_metric(row['Train Accuracy'])} | "
            f"{format_metric(row['Validation Accuracy'])} | {format_metric(row['Test Accuracy'])} | "
            f"{format_metric(row['Balanced Accuracy'])} | {format_metric(row['Precision'])} | "
            f"{format_metric(row['Recall'])} | {format_metric(row['F1-score'])} |"
        )
    return "\n".join(lines)


def build_interpretation(results_df: pd.DataFrame) -> str:
    best = results_df.iloc[0]
    gaps = []
    for _, row in results_df.iterrows():
        gaps.append(
            f"{row['CNN Architecture']}: train-test accuracy gap={row['Train Accuracy'] - row['Test Accuracy']:.4f}"
        )
    gap_text = "; ".join(gaps)
    return (
        f"The best overall model by test F1-score and balanced accuracy is **{best['CNN Architecture']}** "
        f"(F1={best['F1-score']:.4f}, balanced accuracy={best['Balanced Accuracy']:.4f}). "
        f"Overfitting should be judged cautiously from the train/validation/test gaps: {gap_text}. "
        "Only four parsed subjects are available, with validation and test each containing one subject, "
        "so evaluation should be interpreted cautiously. LOSO folds exist as a follow-up sensitivity check "
        "but are not run in this initial notebook."
    )


def write_initial_summary(results_df: pd.DataFrame) -> Path:
    table = markdown_results_table(results_df)
    interpretation = build_interpretation(results_df)
    text = f"""# NTHUDDD2 Kaggle Initial Experiment Summary

## Task

- Task type: Image Classification
- Dataset: Kaggle extracted-frame JPG version of NTHUDDD2
- Protocol note: This is **not** the official raw-video NTHU-DDD benchmark protocol.
- Baseline: single-frame binary drowsiness image classification.

## Labels

- `notdrowsy = 0`
- `drowsy = 1`

## Dataset Size

- Total images: 66,521
- Drowsy: 36,030
- Not drowsy: 30,491
- Parsed subjects: 4 (`001`, `002`, `005`, `006`)

## Input

- Full-frame JPG images
- Dynamic resize to `224 x 224` during training
- No permanent image resizing
- No MTCNN face cropping

## Split

- Subject-level split
- Train subjects: `001`, `005`
- Validation subject: `002`
- Test subject: `006`
- Subject leakage: 0

## Training Settings

- Framework: PyTorch
- Pretrained weights: torchvision ImageNet weights
- Architectures: ResNet18, MobileNetV2, EfficientNet-B0
- Optimizer: Adam
- Learning rate: 1e-4
- Batch size: 32
- Max epochs: 8
- Freeze first epoch: yes
- Early stopping patience: 2
- Loss: weighted cross entropy
- Scheduler: ReduceLROnPlateau
- Positive class for precision/recall/F1: `drowsy`

## Results

{table}

## Interpretation

{interpretation}

## Limitations

- This is the Kaggle extracted-frame JPG version of NTHUDDD2.
- This is not the official raw-video NTHU-DDD benchmark protocol.
- Only 4 parsed subjects are available locally.
- Fixed validation and test splits each contain only one subject.
- The current baseline is single-frame binary drowsiness classification.
- It does not model temporal drowsiness dynamics.
- It does not use official multi-task head, mouth, or eye labels.
- LOSO is recommended later as a sensitivity check.
"""
    out_path = REPORTS_DIR / "initial_experiment_summary.md"
    out_path.write_text(text, encoding="utf-8")
    return out_path


if RUN_WRITE_INITIAL_SUMMARY:
    results_df = build_results_table(all_metrics)
    results_csv = RESULTS_DIR / "initial_results.csv"
    results_df.to_csv(results_csv, index=False)

    metrics_summary = {
        "dataset": "NTHUDDD2 Kaggle extracted-frame JPG",
        "not_official_raw_video_protocol": True,
        "labels": CLASS_TO_ID,
        "expected_counts": EXPECTED_COUNTS,
        "subjects": sorted(manifest_df["subject_id"].unique()),
        "fixed_split_subjects": {k: sorted(v) for k, v in EXPECTED_SUBJECTS.items()},
        "results": results_df.to_dict(orient="records"),
        "model_metrics": all_metrics,
    }
    save_json(metrics_summary, RESULTS_DIR / "metrics_summary.json")
    summary_path = write_initial_summary(results_df)

    print(f"Saved results CSV: {results_csv}")
    print(f"Saved metrics summary: {RESULTS_DIR / 'metrics_summary.json'}")
    print(f"Saved markdown report: {summary_path}")
    print("\nFinal results table:")
    print(markdown_results_table(results_df))
    if display is not None and Markdown is not None:
        display(Markdown(markdown_results_table(results_df)))
else:
    print(
        "Skipping full initial_results.csv / metrics_summary.json rewrite during the "
        "ResNet18 diagnostic pass. Set RUN_WRITE_INITIAL_SUMMARY = True after you "
        "intentionally run the full model comparison."
    )


## Final Notes

All result artifacts are saved under:

`/content/drive/MyDrive/Drowsiness_Detection_Colab/outputs/nthuddd2_kaggle/`

Run this notebook with a GPU runtime for practical training speed. LOSO folds are available for later sensitivity analysis but are intentionally not executed here.